# Heston calibration and structured-product workflow

This notebook contains market inputs and orchestration only. Reusable logic is
in `heston_calibration.py` and `heston_pricing.py`; the volatility workbooks are
kept beside this notebook. Run it with the **spyder-env** kernel and use the
notebook directory as the working directory.

Historical regularization is configured independently for each index below.
With `use_historical=True`, each calibration reloads the latest saved CSV row
for its index and errors if none exists. Set `use_historical=False` explicitly
to calibrate without reading or requiring history.


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

from heston_calibration import (
    DividendCarryCurve,
    calibrate_heston_gatheral,
    historical_parameter_report,
    load_vol_surface,
    run_no_arbitrage_checks,
)
from heston_pricing import (
    calculate_greeks,
    price_autocallable_heston,
    price_phoenix_heston,
    price_snowball_heston,
)


In [ ]:
DATA_DIR = Path.cwd()
PARAM_HISTORY_FILE = DATA_DIR / "heston_calibrated_params.csv"
CALIBRATION_DATE = pd.Timestamp.today().date().isoformat()
PARAMETER_NAMES = ["v0", "kappa", "theta", "sigma", "rho"]
PARAM_HISTORY_COLUMNS = ["calibration_date", "index", *PARAMETER_NAMES]

def read_parameter_history():
    """Read validated CSV history; an absent or zero-byte file has no rows."""
    if not PARAM_HISTORY_FILE.is_file() or PARAM_HISTORY_FILE.stat().st_size == 0:
        return pd.DataFrame(columns=PARAM_HISTORY_COLUMNS)

    history = pd.read_csv(PARAM_HISTORY_FILE)
    missing = [name for name in PARAM_HISTORY_COLUMNS if name not in history.columns]
    if missing:
        raise ValueError(f"{PARAM_HISTORY_FILE} is missing columns: {missing}")

    history = history[PARAM_HISTORY_COLUMNS].copy()
    history["calibration_date"] = pd.to_datetime(
        history["calibration_date"], errors="raise"
    ).dt.normalize()
    if history["calibration_date"].isna().any():
        raise ValueError(f"{PARAM_HISTORY_FILE} contains missing calibration dates")
    for name in PARAMETER_NAMES:
        history[name] = pd.to_numeric(history[name], errors="raise")
    if not np.isfinite(history[PARAMETER_NAMES].to_numpy(dtype=float)).all():
        raise ValueError(f"{PARAM_HISTORY_FILE} historical parameters must be finite")
    return history


def latest_historical_params(index_name):
    """Return the latest CSV parameters for this index, or None if absent."""
    history = read_parameter_history()
    matching = history.loc[history["index"] == index_name].sort_values(
        "calibration_date"
    )
    if matching.empty:
        return None

    latest = matching.iloc[-1]
    print(
        f"Loaded {index_name} historical parameters from "
        f"{latest['calibration_date'].date()} in {PARAM_HISTORY_FILE.name}."
    )
    return {name: float(latest[name]) for name in PARAMETER_NAMES}

## Market and calibration configuration

`history_penalty_iv=0.002` makes an average one-tolerance parameter move cost
the same as 0.2 volatility points of IV RMSE. The default tolerances allow more
movement in `v0` than in the structural parameters. The historical point is
also supplied to differential evolution when `warm_start_from_history=True`.


In [ ]:
DATA_DIR = Path.cwd()
required_files = ["vol_surface1000.xlsx", "vol_surface300.xlsx"]
missing = [name for name in required_files if not (DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing}. Set DATA_DIR to the equity folder containing this notebook."
    )

# Historical calibration reloads the required CSV reference when invoked.
INDEX_CONFIG = {
    "CSI 1000": {
        "index_name": "CSI 1000",
        "surface": "vol_surface1000.xlsx",
        "spot": 6427.17,
        "rate": 0.014,
        "effective_yields": np.array(
            [0.0758, 0.0905, 0.0936, 0.0991,0.0898]
        ),
        "fix_kappa": False,
        "kappa_fixed": 8.0,
        "stage1_fixed_rho": -0.2,
        "stage2_penalty": 1.0,
        "weight_by_T": True,
        "use_historical": True,
        "historical_params": None,
        "history_penalty_iv": 0.002,
        "history_tolerances": None,
        "warm_start_from_history": True,
        "use_historical_rho_in_stage1": True,
        "seed": 2026
    },
    "CSI 300": {
        "index_name": "CSI 300",
        "surface": "vol_surface300.xlsx",
        "spot": 4049.55,
        "rate": 0.014,
        "effective_yields": np.array(
            [0.0854, 0.0969, 0.0900, 0.0866,0.0787]
        ),
        "fix_kappa": False,
        "kappa_fixed": 7.0,
        "stage1_fixed_rho": -0.2,
        "stage2_penalty": 1.0,
        "weight_by_T": True,
        "use_historical": True,
        "historical_params": None,
        "history_penalty_iv": 0.002,
        "history_tolerances": None,
        "warm_start_from_history": True,
        "use_historical_rho_in_stage1": True,
        "seed": 2026
    },
}

config_1000 = INDEX_CONFIG["CSI 1000"]
config_300 = INDEX_CONFIG["CSI 300"]
params_1000 = (latest_historical_params("CSI 1000")
               if config_1000["use_historical"] else None)
params_300 = (latest_historical_params("CSI 300")
               if config_300["use_historical"] else None)

In [ ]:
# Optional CSV inspection; uncomment to run explicitly.
# latest_historical_params("CSI 300")

In [ ]:
def calibrate_index(config):
    """Resolve the current history requirement before any market or optimizer work."""
    config["historical_params"] = None
    if config["use_historical"]:
        index_name = config["index_name"]
        historical = latest_historical_params(index_name)
        if historical is None:
            raise ValueError(
                f"No saved parameters for {index_name} in {PARAM_HISTORY_FILE}. "
                "Provide saved parameters or explicitly set use_historical=False."
            )
        config["historical_params"] = historical
    market = load_vol_surface(DATA_DIR / config["surface"])
    maturities = np.sort(market["T"].unique())
    curve = DividendCarryCurve(maturities, config["effective_yields"])
    params, fitted = calibrate_heston_gatheral(
        market,
        config["rate"],
        curve,
        config["spot"],
        fix_kappa=config["fix_kappa"],
        kappa_fixed=config["kappa_fixed"],
        stage1_fixed_rho=config["stage1_fixed_rho"],
        stage2_penalty=config["stage2_penalty"],
        weight_by_T=config["weight_by_T"],
        use_historical=config["use_historical"],
        historical_params=config["historical_params"],
        history_penalty_iv=config["history_penalty_iv"],
        history_tolerances=config["history_tolerances"],
        warm_start_from_history=config["warm_start_from_history"],
        use_historical_rho_in_stage1=config["use_historical_rho_in_stage1"],
        seed = 123,
    )
    return market, curve, params, fitted


def show_calibration_summary(config, params, fitted):
    print("Calibrated parameters:")
    for name, value in params.items():
        print(f"  {name}: {value:.10f}")

    if config["use_historical"]:
        active_names = ["v0", "kappa", "theta", "sigma", "rho"]
        if config["fix_kappa"]:
            active_names.remove("kappa")
        print("\nMovement from historical parameters:")
        print(
            historical_parameter_report(
                params,
                config["historical_params"],
                history_tolerances=config["history_tolerances"],
                active_names=active_names,
            ).to_string(index=False)
        )

    print("\nCalibration diagnostics:")
    print(fitted.attrs["calibration_diagnostics"])


## CSI 1000 calibration

In [ ]:
config_1000 = INDEX_CONFIG["CSI 1000"]
market_1000, q_curve_1000, params_1000, fitted_1000 = calibrate_index(
    config_1000
)

print(q_curve_1000.knot_table(S0=config_1000["spot"], r=config_1000["rate"]))
show_calibration_summary(config_1000, params_1000, fitted_1000)
fitted_1000.head(10)


In [ ]:
run_no_arbitrage_checks(
    market_1000,
    params_1000,
    config_1000["spot"],
    config_1000["rate"],
    q_curve_1000,
    "000852 CSI 1000",
)

## CSI 300 calibration

In [ ]:
config_300 = INDEX_CONFIG["CSI 300"]
market_300, q_curve_300, params_300, fitted_300 = calibrate_index(config_300)

print(q_curve_300.knot_table(S0=config_300["spot"], r=config_300["rate"]))
show_calibration_summary(config_300, params_300, fitted_300)
fitted_300.head(10)


In [ ]:
params_300

In [ ]:
run_no_arbitrage_checks(
    market_300,
    params_300,
    config_300["spot"],
    config_300["rate"],
    q_curve_300,
    "000300 CSI 300",
)

## Snowball pricing and Greeks example

In [ ]:
spot = config_1000["spot"]
obs_times = np.arange(21, 21 * 19 + 1, 21)  # trading days
pay_times = np.arange(180, 30 * 24 + 1, 30)  # 30/360 calendar days

snowball_inputs = {
    "S0": spot,
    "T": 1.0,
    "coupon": .2401,
    "knock_out": spot,
    "H_in": 0.85 * spot,
    "obs_times": obs_times,
    "r": config_1000["rate"],
    "q": q_curve_1000,
    "kappa": params_1000["kappa"],
    "theta": params_1000["theta"],
    "sigma_v": params_1000["sigma"],
    "rho": params_1000["rho"],
    "v0": params_1000["v0"],
    "Ns": 1024,
    "Nv": 40,
    "steps_per_day": 16,
    "daily_ki": True,
    "SPD": 0.8,
    "pay_times": pay_times,
}
M = 1000000
snowball_price = price_snowball_heston(**snowball_inputs)
snowball_greeks = calculate_greeks(price_snowball_heston, snowball_inputs)
print(f"Price per unit notional: {snowball_price * M:.8f}")
print(f"delta: {snowball_greeks['delta'] * M:.8f}")
snowball_greeks

In [ ]:
spot = config_300["spot"]
obs_times = np.arange(21, 21 * 19 + 1, 21)  # trading days
pay_times = np.arange(180, 30 * 24 + 1, 30)  # 30/360 calendar days

snowball_inputs = {
    "S0": 4049.55, #spot,
    "K": 4866.49 * 0.9,
    "T": 399/252,
    "coupon": 0.02,
    "knock_out": 4866.49 ,
    "H_in": 0.67 * 4866.49,
    "obs_times": obs_times,
    "r": config_300["rate"],
    "q": config_300["effective_yields"],
    "q_maturities":np.array([40,59,99,162,226])/252,
    "kappa": params_300["kappa"],
    "theta": params_300["theta"],
    "sigma_v": params_300["sigma"],
    "rho": params_300["rho"],
    "v0": params_300["v0"],
    "Ns": 2048,
    "Nv": 160,
    "steps_per_day": 16,
    "daily_ki": True,
    "SPD": 0,
    "pay_times": pay_times,
}
M = 10000000
snowball_price = price_snowball_heston(**snowball_inputs)
snowball_greeks = calculate_greeks(price_snowball_heston, snowball_inputs)
print(f"Price per unit notional: {snowball_price * M:.8f}")
print(f"delta: {snowball_greeks['delta'] * M:.8f}")
snowball_greeks

In [ ]:
spot = config_1000["spot"]
obs_times = np.arange(21, 21 * 24 + 1, 21)  # trading days
pay_times = np.arange(30, 30 * 24 + 1, 30)  # 30/360 calendar days

autocallable_inputs = {
    "S0": spot,
    "T": 2.0,
    "coupon": 0.0286,
    "knock_out": spot * 1.01,
    "obs_times": obs_times,
    "premium": 0.01,
    "r": config_1000["rate"],
    "q": q_curve_1000,
    "kappa": params_1000["kappa"],
    "theta": params_1000["theta"],
    "sigma_v": params_1000["sigma"],
    "rho": params_1000["rho"],
    "v0": params_1000["v0"],
    "Ns": 1000,
    "Nv": 40,
    "steps_per_day": 1,
    "pay_times": pay_times,
    "mg": 0,
    "mg_ist": None,
    "q_maturities": None,
}
M = 10000000
auto_price = price_autocallable_heston(**autocallable_inputs)
snowball_greeks = calculate_greeks(price_autocallable_heston, autocallable_inputs)
print(f"Price per unit notional: {auto_price * M:.8f}")
print(f"delta: {snowball_greeks['delta'] * M:.8f}")
snowball_greeks

In [ ]:
spot = config_300["spot"]
obs_times = np.arange(21, 21 * 12 + 1, 21)  # trading days
pay_times = np.arange(30, 30 * 12 + 1, 30)  # 30/360 calendar days

autocallable_inputs = {
    "S0": spot,
    "T": 1.0,
    "coupon": 0.0265,
    "knock_out": spot * 1.01,
    "obs_times": obs_times,
    "premium": 0.01,
    "r": config_300["rate"],
    "q": 0.06,#q_curve_300,
    "kappa": params_300["kappa"],
    "theta": params_300["theta"],
    "sigma_v": params_300["sigma"],
    "rho": params_300["rho"],
    "v0": params_300["v0"],
    "Ns": 1000,
    "Nv": 40,
    "steps_per_day": 1,
    "pay_times": pay_times,
    "mg": 0,
    "mg_ist": None,
    "q_maturities": None,
}
M = 10000000
auto_price = price_autocallable_heston(**autocallable_inputs)
snowball_greeks = calculate_greeks(price_autocallable_heston, autocallable_inputs)
print(f"Price per unit notional: {auto_price * M:.8f}")
print(f"delta: {snowball_greeks['delta'] * M:.8f}")
snowball_greeks

In [ ]:
spot = config_1000["spot"]
obs_times = np.arange(21, 21 * 24 + 1, 21)/252  # years (trading days /252)

phoenix_inputs = {
    "S0":spot,
    "T": 2.,
    "coupon": 0.0067,
    "knock_out": spot,
    "H_in": 0.8*spot,
    "obs_times": obs_times,
   "r": config_1000["rate"],
    "q": 0.07,
    "kappa": params_1000["kappa"],
    "theta": params_1000["theta"],
    "sigma_v": params_1000["sigma"],
    "rho": params_1000["rho"],
    "v0": 0.06,#params_1000["v0"],
    "Ns": 1000,
    "Nv": 40,
    "steps_per_day": 1,
    "H_coupon": None,
    "daily_ki": False,
    "SPD": 0.8,
    "mg": 0,
    "mg_ist": None,
    "full_term":None,
    "K":None,
    "q_maturities":None,
}
M = 30000000
phoe_price = price_phoenix_heston(**phoenix_inputs)
# snowball_greeks = calculate_greeks(price_snowball_heston, snowball_inputs)
print(f"Price per unit notional: {phoe_price * M:.8f}")
# snowball_greeks

In [ ]:
spot = config_300["spot"]
obs_times = np.arange(21, 21 * 6 + 1, 21)/252  # years (trading days /252)

phoenix_inputs = {
    "S0":spot,
    "T": .5,
    "coupon": 0.0068,
    "knock_out": spot,
    "H_in": 0.85*spot,
    "obs_times": obs_times,
   "r": config_300["rate"],
    "q": 0.06,#q_curve_300,
    "kappa": params_300["kappa"],
    "theta": params_300["theta"],
    "sigma_v": params_300["sigma"],
    "rho": params_300["rho"],
    "v0": params_300["v0"],
    "Ns": 1000,
    "Nv": 40,
    "steps_per_day": 1,
    "H_coupon": None,
    "daily_ki": True,
    "SPD": 0.85,
    "mg": 0,
    "mg_ist": None,
    "full_term":None,
    "K":None,
    "q_maturities":None,
}
M = 3000000
phoe_price = price_phoenix_heston(**phoenix_inputs)
# snowball_greeks = calculate_greeks(price_snowball_heston, snowball_inputs)
print(f"Price per unit notional: {phoe_price * M:.8f}")
# snowball_greeks